# Stage C 03i — inexpensive c16 decoding ablation

This factorized screen changes no model weights. It isolates temperature, top-k, and top-p effects using two held-out prompts and 512 generated tokens per policy. Temperature 0.6 is included. Nine unique policies replace a wasteful 48-condition Cartesian grid, and every policy is compared with matched real continuations.

In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='PIN_AFTER_COMMIT'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
SOURCE_RUN='c16_deep_adaptive_5m_paper_exact'
OUTPUT_NAME='decoding_ablation_v1'
TAXONOMY_MANIFEST=f'{DRIVE_ROOT}/stage_c_dataset/manifests/accession_manifest.parquet'
PROMPTS=2; PROMPT_TOKENS=32; NEW_TOKENS=512; SEED=20260781
# label, temperature, top-k ('none' means unrestricted), top-p
POLICIES=[
 ('temp_0p6_k1024_p099',0.6,'1024',0.99),
 ('temp_0p8_k1024_p099',0.8,'1024',0.99),
 ('temp_1p0_k1024_p099',1.0,'1024',0.99),
 ('temp_1p1_k1024_p099',1.1,'1024',0.99),
 ('temp_0p8_k128_p099',0.8,'128',0.99),
 ('temp_0p8_k512_p099',0.8,'512',0.99),
 ('temp_0p8_knone_p099',0.8,'none',0.99),
 ('temp_0p8_knone_p095',0.8,'none',0.95),
 ('temp_0p8_knone_p100',0.8,'none',1.0),
]


In [ ]:
from pathlib import Path
from google.colab import drive
import csv, json, shutil, subprocess, sys
mount=Path('/content/drive')
if not (mount/'MyDrive').is_dir(): drive.mount(str(mount),timeout_ms=120000)
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Select a Colab T4-or-better GPU runtime.')
if not shutil.which('prodigal'):
    subprocess.run(['apt-get','update'],check=True); subprocess.run(['apt-get','install','-y','prodigal'],check=True)
selection=json.loads((Path(DRIVE_ROOT)/'runs/c1_tokenizers_cpu/tokenizer_selection.json').read_text())
dataset=Path(DRIVE_ROOT)/'stage_c_dataset/ordered_streams'/selection['selected_tokenizer']
source_dir=Path(DRIVE_ROOT)/'runs'/SOURCE_RUN; checkpoint=source_dir/'latest.pt'; root=source_dir/OUTPUT_NAME
PROTOCOL=repo/'studies/stage_c_ecoli_escherichia_paper_deep_memory_v2/protocol.json'
AMENDMENT=repo/'studies/stage_c_ecoli_escherichia_paper_deep_memory_v2/amendments/c16_decoding_ablation_v1.json'
STUDY_ROOT=Path(DRIVE_ROOT)/'study/stage_c_ecoli_escherichia_paper_deep_memory_v2'
for required in (dataset,checkpoint,Path(TAXONOMY_MANIFEST),PROTOCOL,AMENDMENT):
    if not required.exists(): raise FileNotFoundError(required)
subprocess.run(['seqtrainer-titans-stage-c-study','initialize','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT)],check=True)
source=json.loads(AMENDMENT.read_text()); drive_amendment=STUDY_ROOT/'amendments'/f"{source['amendment_id']}.json"
if not drive_amendment.exists():
    subprocess.run(['seqtrainer-titans-stage-c-study','amend','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT),'--amendment-id',source['amendment_id'],'--rationale',source['rationale'],'--classification',source['classification'],'--expected-impact',source['expected_impact'],'--changes',json.dumps(source['changes'],sort_keys=True)],check=True)
print('GPU:',torch.cuda.get_device_name(0)); print('Ablation root:',root)


In [ ]:
def run_logged(log_dir,label,command):
    try:
        subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',str(log_dir),'--label',label,'--repo',str(repo),'--',*command],check=True)
    except subprocess.CalledProcessError:
        for path in (log_dir/'FAILED.txt',log_dir/'logs'/f'{label}.log'):
            if path.exists(): print(path.read_text(errors='replace')[-16000:])
        raise
for index,(label,temperature,top_k,top_p) in enumerate(POLICIES,1):
    output=root/label; report=output/'generation_evaluation.json'
    if report.exists():
        print(f'[{index}/{len(POLICIES)}] completed; skipping {label}')
        continue
    print(f'[{index}/{len(POLICIES)}] running {label}: T={temperature}, k={top_k}, p={top_p}')
    command=['seqtrainer-titans-stage-c-generate','--dataset-dir',str(dataset),'--taxonomy-manifest',TAXONOMY_MANIFEST,'--checkpoint',str(checkpoint),'--output-dir',str(output),'--split','val','--species','Escherichia coli','--prompts',str(PROMPTS),'--prompt-tokens',str(PROMPT_TOKENS),'--new-tokens',str(NEW_TOKENS),'--temperatures',str(temperature),'--top-k',top_k,'--top-p',str(top_p),'--seed',str(SEED),'--device','cuda','--memory-mode','adaptive','--prodigal',shutil.which('prodigal'),'--protocol',str(PROTOCOL),'--protocol-amendment',str(AMENDMENT),'--run-id','adaptive_exploration_5m_decoding_ablation']
    run_logged(output,f'generate_{label}',command)


In [ ]:
rows=[]
for label,temperature,top_k,top_p in POLICIES:
    result=json.loads((root/label/'generation_evaluation.json').read_text())
    group=next(key for key in result['distribution_summary'] if key!='reference')
    generated=result['distribution_summary'][group]; reference=result['distribution_summary']['reference']
    gp=result['prodigal']['groups'][group]; rp=result['prodigal']['groups']['reference']
    rows.append({'policy':label,'temperature':temperature,'top_k':top_k,'top_p':top_p,'gc':generated['gc_fraction'],'gc_abs_error':abs(generated['gc_fraction']-reference['gc_fraction']),'base_entropy_bits':generated['base_entropy_bits'],'max_homopolymer':generated['max_homopolymer'],'overlapping_6mer_diversity_ratio':generated['unique_6mer_fraction']/reference['unique_6mer_fraction'],'aligned_6mer_diversity_ratio':generated['aligned_unique_6mer_fraction']/reference['aligned_unique_6mer_fraction'],'jsd_3mer':result['kmer_jsd_to_heldout_reference'][group]['3'],'jsd_6mer':result['kmer_jsd_to_heldout_reference'][group]['6'],'orfs_90bp_ratio':generated['heuristic_orfs_at_least_90bp']/max(reference['heuristic_orfs_at_least_90bp'],1),'longest_orf_ratio':generated['heuristic_longest_orf_bases']/max(reference['heuristic_longest_orf_bases'],1),'prodigal_genes_per_10kb_ratio':gp['genes_per_10kb']/max(rp['genes_per_10kb'],1e-12),'prodigal_coding_density_ratio':gp['coding_density']/max(rp['coding_density'],1e-12),'prodigal_median_gene_ratio':gp['median_gene_bases']/max(rp['median_gene_bases'],1e-12),'prodigal_median_intergenic_ratio':gp['median_intergenic_bases']/max(rp['median_intergenic_bases'],1e-12)})
rows.sort(key=lambda row:(row['jsd_6mer'],-row['aligned_6mer_diversity_ratio'],row['gc_abs_error']))
root.mkdir(parents=True,exist_ok=True)
(root/'decoding_ablation_summary.json').write_text(json.dumps({'classification':'exploratory_decoding_ablation','ranking_rule':'ascending 6-mer JSD, then descending aligned diversity ratio, then ascending GC absolute error','rows':rows},indent=2,sort_keys=True)+'\n')
with (root/'decoding_ablation_summary.csv').open('w',newline='') as handle:
    writer=csv.DictWriter(handle,fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)
lines=['# c16 decoding-ablation screen','','This is an exploratory ordering, not a biological quality score. Confirm finalists with four prompts and 1,024 tokens.','','| Rank | Policy | GC error | aligned diversity / ref | 3-mer JSD | 6-mer JSD | genes/10kb / ref | coding density / ref |','|---:|---|---:|---:|---:|---:|---:|---:|']
for rank,row in enumerate(rows,1): lines.append(f"| {rank} | {row['policy']} | {row['gc_abs_error']:.4f} | {row['aligned_6mer_diversity_ratio']:.3f} | {row['jsd_3mer']:.4f} | {row['jsd_6mer']:.4f} | {row['prodigal_genes_per_10kb_ratio']:.3f} | {row['prodigal_coding_density_ratio']:.3f} |")
(root/'DECODING_ABLATION_REPORT.md').write_text('\n'.join(lines)+'\n')
marker=STUDY_ROOT/'record_markers/c16_decoding_ablation_v1.json'
if not marker.exists():
    subprocess.run(['seqtrainer-titans-stage-c-study','record','--protocol',str(PROTOCOL),'--protocol-amendment',str(AMENDMENT),'--study-root',str(STUDY_ROOT),'--run-id','adaptive_exploration_5m_decoding_ablation','--evidence-tier','exploratory','--artifact',str(root)],check=True)
    marker.parent.mkdir(parents=True,exist_ok=True); marker.write_text(json.dumps({'run_id':'adaptive_exploration_5m_decoding_ablation','artifact':str(root)},indent=2)+'\n')
subprocess.run(['seqtrainer-titans-stage-c-study','verify','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT)],check=True)
print((root/'DECODING_ABLATION_REPORT.md').read_text())
print('Full summary:',root/'decoding_ablation_summary.json')
print('Do not select from two prompts alone; review this screen, then confirm the best 2–3 policies at the original 03h scale.')
